# Module 6: Building RAG — Day 1
## From Documents to Intelligent Answers

**Course:** AI Agent Engineering | **Week:** 3 | **Duration:** 150 minutes

---

## Learning Objectives

By the end of this notebook, you will be able to:
1. Load and parse policy documents using LangChain document loaders
2. Chunk documents using configurable fixed-size splitting
3. Generate semantic embeddings with OpenAI's embedding API
4. Index and query a ChromaDB vector store
5. Build a grounded LLM answer pipeline using retrieved context
6. Evaluate retrieval quality using hit-rate metrics

## Prerequisites

- Completed Module 5: Advanced Prompt Engineering
- OpenAI API key in `.env` file as `OPENAI_API_KEY`
- Install packages: `pip install chromadb openai langchain langchain-community tiktoken pypdf python-dotenv`

---

## Setup: Imports and Configuration

In [ ]:
# Install dependencies (run once)
# !pip install chromadb openai langchain langchain-community tiktoken pypdf python-dotenv

In [ ]:
import os
import time
import logging
from pathlib import Path

import chromadb
from openai import OpenAI
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from dotenv import load_dotenv

load_dotenv()

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

print("✓ Setup complete")
print(f"  OpenAI key set: {'Yes' if os.environ.get('OPENAI_API_KEY') else 'NO — add to .env file'}")

---

## Section 1: The Business Problem

### Why We're Building This

You're building a **Policy QnA Bot** for Vidvatta University's HR team.

**Current situation:**
- HR receives 200+ policy questions per week via email
- Staff manually search through 50+ page PDF handbooks
- Answers are inconsistent — different people quote different versions
- New employees especially struggle

**Your success criteria:**
| Metric | Target |
|--------|--------|
| Correct answer rate | > 85% |
| Response latency | < 3 seconds |
| Source traceability | 100% (every answer cites a source) |
| Edge case handling | Empty queries, out-of-scope questions |

**Key question before we write code:** What does a *wrong* answer cost in this context?

In [ ]:
# Create sample policy document for the lab
SAMPLE_POLICY = """
HR POLICY DOCUMENT - VIDVATTA UNIVERSITY

SECTION 1: ANNUAL LEAVE
Full-time employees are entitled to 20 days of annual leave per calendar year.
Part-time employees receive annual leave on a pro-rata basis proportional to their contracted hours.
Annual leave must be approved by the line manager at least 5 working days in advance.
Unused leave may be carried over to the following year up to a maximum of 5 days.

SECTION 2: SICK LEAVE
Employees are entitled to 10 days of paid sick leave per year.
A medical certificate is required for any absence exceeding 3 consecutive days.
Sick leave does not carry over to the following year.
Employees must notify their manager before 9:00 AM on the first day of absence.

SECTION 3: MATERNITY AND PATERNITY LEAVE
Maternity leave entitlement is 26 weeks, with the first 16 weeks at full pay.
Paternity leave entitlement is 2 weeks at full pay, to be taken within 8 weeks of birth.
Adoption leave follows the same entitlement structure as maternity leave.
Employees must give 8 weeks written notice before commencing parental leave.

SECTION 4: RESIGNATION AND NOTICE PERIOD
Employees with less than 2 years of service must provide 4 weeks notice.
Employees with 2 or more years of service must provide 8 weeks notice.
Notice must be submitted in writing to the HR department and the line manager.
Garden leave may be applied at the discretion of the university.

SECTION 5: DISCIPLINARY PROCEDURE
Step 1: Verbal warning documented in writing and placed on the employee file.
Step 2: Written warning issued and signed by HR, employee, and line manager.
Step 3: Final written warning with a 6-month review period.
Step 4: Dismissal with right of appeal to the HR Director within 10 working days.
All disciplinary hearings require 48 hours advance notice to the employee.

SECTION 6: REMOTE WORK POLICY
Employees may work remotely up to 2 days per week with manager approval.
Core hours of 10:00 AM to 3:00 PM must be observed regardless of work location.
Remote work arrangements are reviewed annually and may be revoked with 2 weeks notice.
Equipment for remote work is provided by the university for the first year.
"""

# Save to disk
policy_path = Path("sample_hr_policy.txt")
policy_path.write_text(SAMPLE_POLICY, encoding="utf-8")
print(f"✓ Sample policy document created: {policy_path} ({len(SAMPLE_POLICY)} chars)")

---

## Section 2: Document Ingestion

### Concept: Document Loaders

LangChain provides loaders for common document formats. Each loader returns a list of `Document` objects, where each document has:
- `page_content`: the raw text
- `metadata`: source info (filename, page number, etc.)

**Why metadata matters:** It's the audit trail. Every retrieved chunk must be traceable to a source.

In [ ]:
# Load the document
loader = TextLoader(str(policy_path), encoding="utf-8")
documents = loader.load()

print(f"Loaded {len(documents)} document(s)")
print(f"\nFirst document preview:")
print(f"  Content (first 300 chars): {documents[0].page_content[:300]}...")
print(f"  Metadata: {documents[0].metadata}")

---

## Section 3: Text Chunking

### Concept: Why We Chunk

We split documents into smaller chunks because:
1. **Precision:** Small, focused chunks embed one idea → better retrieval
2. **Context limits:** LLMs can only receive limited context per call
3. **Cost:** Only send relevant chunks, not entire documents

**`RecursiveCharacterTextSplitter`** tries to split on natural boundaries first (`\n\n`, `\n`, `. `) before falling back to character splitting.

In [ ]:
# Create text splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,        # max characters per chunk
    chunk_overlap=50,      # overlap to preserve boundary context
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = splitter.split_documents(documents)

print(f"Total chunks created: {len(chunks)}")
print(f"\nSample chunks:")
for i, chunk in enumerate(chunks[:3]):
    print(f"\n  --- Chunk {i} ({len(chunk.page_content)} chars) ---")
    print(f"  {chunk.page_content[:250]}")

In [ ]:
# Token counting for chunks
try:
    import tiktoken
    enc = tiktoken.encoding_for_model("text-embedding-3-small")
    token_counts = [len(enc.encode(c.page_content)) for c in chunks]
    print(f"Token distribution across {len(chunks)} chunks:")
    print(f"  Min:  {min(token_counts)} tokens")
    print(f"  Max:  {max(token_counts)} tokens")
    print(f"  Avg:  {sum(token_counts) // len(token_counts)} tokens")
except ImportError:
    print("tiktoken not installed — skipping token count")

---

## Section 4: Document Embeddings

### Concept: How Text Becomes a Vector

An embedding model reads text and outputs a **list of numbers** (a vector) that encodes the text's *semantic meaning*.

```
"annual leave entitlement"  →  [0.12, -0.87, 0.34, 0.55, ...]  (1536 numbers)
```

**The key property:** Semantically similar texts produce vectors that are *close together* in this 1536-dimensional space. This is what makes semantic search possible — we're searching by meaning, not keywords.

### Why `text-embedding-3-small`?
- 1536 dimensions (good balance of quality and cost)
- ~$0.02 per million tokens (very affordable)
- Excellent performance for English policy/FAQ content

In [ ]:
# Generate a single embedding
sample_text = "How many annual leave days do employees get?"

response = client.embeddings.create(
    input=sample_text,
    model="text-embedding-3-small"
)

sample_embedding = response.data[0].embedding
print(f"Input text: '{sample_text}'")
print(f"Embedding dimensions: {len(sample_embedding)}")
print(f"First 10 values: {[round(v, 4) for v in sample_embedding[:10]]}")

In [ ]:
# Demonstrate semantic similarity with cosine similarity
import math

def cosine_similarity(v1: list, v2: list) -> float:
    dot = sum(a * b for a, b in zip(v1, v2))
    norm1 = math.sqrt(sum(a**2 for a in v1))
    norm2 = math.sqrt(sum(b**2 for b in v2))
    return dot / (norm1 * norm2)

texts = [
    "How many annual leave days do employees get?",
    "What is the vacation day entitlement for staff?",  # semantically similar
    "What is the disciplinary procedure for misconduct?",  # different topic
    "What is the salary pay grade structure?",  # unrelated
]

embeddings = [
    client.embeddings.create(input=t, model="text-embedding-3-small").data[0].embedding
    for t in texts
]

print("Cosine similarity to: 'How many annual leave days do employees get?'")
print("-" * 70)
for i, (text, emb) in enumerate(zip(texts, embeddings)):
    score = cosine_similarity(embeddings[0], emb)
    bar = "#" * int(score * 40)
    print(f"{score:.4f}  {bar}")
    print(f"        '{text[:60]}'")

**Observe:** The semantically similar text (vacation days) should score much higher than the unrelated texts, even though the exact words are different. This is semantic search in action.

---

## Section 5: Vector Store — ChromaDB

### Concept: What a Vector Store Does

A vector database stores embeddings alongside the original text and metadata, and provides **fast similarity search** (finding the k most similar vectors to a query vector).

ChromaDB is a lightweight, embedded vector store — zero setup, runs in-memory or on disk.

In [ ]:
# Create ChromaDB collection
chroma_client = chromadb.Client()

# Delete existing collection if re-running
try:
    chroma_client.delete_collection("hr_policies")
except:
    pass

collection = chroma_client.create_collection(
    name="hr_policies",
    metadata={"hnsw:space": "cosine"}  # use cosine similarity
)

print(f"✓ Created collection: 'hr_policies'")

In [ ]:
# Embed and index all chunks
print(f"Embedding and indexing {len(chunks)} chunks...")
start = time.time()

texts = [c.page_content for c in chunks]
metadatas = [c.metadata for c in chunks]
ids = [f"chunk_{i:05d}" for i in range(len(chunks))]

# Batch embed for efficiency
BATCH_SIZE = 20
all_embeddings = []
for i in range(0, len(texts), BATCH_SIZE):
    batch = texts[i:i + BATCH_SIZE]
    response = client.embeddings.create(input=batch, model="text-embedding-3-small")
    all_embeddings.extend([item.embedding for item in response.data])
    print(f"  Embedded {min(i + BATCH_SIZE, len(texts))}/{len(texts)} chunks")

# Add to ChromaDB
collection.add(
    documents=texts,
    embeddings=all_embeddings,
    metadatas=metadatas,
    ids=ids
)

elapsed = round(time.time() - start, 2)
print(f"\n✓ Indexed {collection.count()} chunks in {elapsed}s")

---

## Section 6: Semantic Retrieval

### Concept: Query → Embed → Search → Return Top-K

At query time:
1. The user's question is embedded using the same model used for indexing
2. ChromaDB finds the k chunks whose vectors are most similar (by cosine similarity)
3. Those chunks are returned with their similarity scores

**ChromaDB note:** It returns *distance* (lower = more similar). Since we're using cosine space, `similarity = 1 - distance`.

In [ ]:
# Run a semantic query
query = "How many annual leave days am I entitled to?"

results = collection.query(
    query_texts=[query],
    n_results=3
)

print(f"Query: '{query}'")
print(f"\nTop {len(results['documents'][0])} results:")
print("-" * 70)

for doc, distance, meta, chunk_id in zip(
    results["documents"][0],
    results["distances"][0],
    results["metadatas"][0],
    results["ids"][0]
):
    similarity = round(1 - distance, 4)
    print(f"ID: {chunk_id} | Score: {similarity}")
    print(f"Text: {doc[:200]}")
    print()

In [ ]:
# Helper function with score threshold guard
def retrieve(query: str, collection, n_results: int = 3, min_score: float = 0.70) -> dict:
    """Retrieve with logging and threshold guard."""

    # Guard: empty query
    if not query or not query.strip():
        return {"error": "Empty query", "documents": [], "scores": []}

    start = time.time()
    results = collection.query(query_texts=[query], n_results=n_results)
    latency_ms = round((time.time() - start) * 1000, 2)

    documents = results["documents"][0]
    scores = [round(1 - d, 4) for d in results["distances"][0]]

    print(f"  Latency: {latency_ms}ms | Top score: {scores[0] if scores else 'N/A'}")

    # Guard: low confidence
    if scores and scores[0] < min_score:
        return {"low_confidence": True, "top_score": scores[0], "documents": [], "scores": []}

    return {
        "documents": documents,
        "scores": scores,
        "metadatas": results["metadatas"][0],
        "ids": results["ids"][0],
        "latency_ms": latency_ms
    }

# Test the helper
test_result = retrieve("What is the sick leave policy?", collection)
print(f"Results: {len(test_result['documents'])} chunks retrieved")

---

## Section 7: Grounded Generation

### Concept: Context-Constrained Prompting

The key instruction that prevents hallucination:
> *"Answer ONLY using the provided context."*

This forces the LLM to be a reader/summarizer, not a knowledge generator. If the context doesn't contain the answer, it should say so.

In [ ]:
def generate_answer(question: str, context: str) -> str:
    """Generate a grounded answer using retrieved context."""
    if not context.strip():
        return "I don't have enough information to answer that question."

    prompt = f"""You are an HR policy assistant. Answer the question using ONLY the provided context.
If the answer is not found in the context, say: "I don't have that information in the policy documents."
Do not infer or guess beyond what is explicitly stated.

Context:
{context}

Question: {question}
Answer:"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return response.choices[0].message.content.strip()


def rag_query(question: str, collection, n_results: int = 3, min_score: float = 0.70) -> dict:
    """Full RAG pipeline: retrieve + generate."""
    retrieval = retrieve(question, collection, n_results, min_score)

    if retrieval.get("error") or retrieval.get("low_confidence"):
        return {
            "question": question,
            "answer": "I don't have reliable information on that topic in the policy documents.",
            "sources": []
        }

    context = "\n\n---\n\n".join(retrieval["documents"])
    answer = generate_answer(question, context)

    sources = [
        {"chunk_id": cid, "score": score, "source": meta.get("source", "unknown")}
        for cid, score, meta in zip(retrieval["ids"], retrieval["scores"], retrieval["metadatas"])
    ]

    return {"question": question, "answer": answer, "sources": sources}


print("✓ RAG pipeline functions defined")

In [ ]:
# Run sample queries through the full pipeline
queries = [
    "How many annual leave days do full-time employees get?",
    "What is the notice period if I resign after 3 years?",
    "How does the disciplinary procedure work?",
    "What is the salary structure?",  # Out of scope — should give 'I don't know'
]

for q in queries:
    print(f"\n{'='*65}")
    print(f"Q: {q}")
    result = rag_query(q, collection)
    print(f"A: {result['answer']}")
    if result["sources"]:
        print(f"Source: {result['sources'][0]['chunk_id']} (score: {result['sources'][0]['score']})")

---

## Section 8: Retrieval Evaluation — Hit-Rate

### Concept: Measuring What Matters

**Hit-rate** = of the chunks that *should* have been retrieved, how many *were* retrieved?

```
Hit-Rate = |Retrieved ∩ Relevant| / |Relevant|
```

This is the single most important metric for your RAG baseline. Before optimizing anything — measure this first.

In [ ]:
# First, discover what chunk IDs correspond to each policy section
# This helps us build the eval set
print("Chunk ID → Content mapping (for building eval set):")
print("-" * 70)

all_results = collection.get(include=["documents", "metadatas"])
for chunk_id, doc in zip(all_results["ids"], all_results["documents"]):
    print(f"\n{chunk_id}:")
    print(f"  {doc[:150]}...")

In [ ]:
# Build evaluation set — update chunk IDs based on the output above
# The eval set maps queries to the ground-truth relevant chunk IDs

# TODO in lab: update relevant_chunk_ids based on the chunk IDs printed above
EVAL_SET = [
    {
        "query": "How many annual leave days do full-time employees get?",
        "relevant_chunk_ids": ["chunk_00000"]  # update with actual ID
    },
    {
        "query": "Do I need a medical certificate for sick leave?",
        "relevant_chunk_ids": ["chunk_00001"]  # update with actual ID
    },
    {
        "query": "How long is maternity leave?",
        "relevant_chunk_ids": ["chunk_00002"]  # update with actual ID
    },
    {
        "query": "What is the notice period for resignation?",
        "relevant_chunk_ids": ["chunk_00003"]  # update with actual ID
    },
    {
        "query": "What are the steps in the disciplinary procedure?",
        "relevant_chunk_ids": ["chunk_00004"]  # update with actual ID
    },
    {
        "query": "Can I work from home and how many days per week?",
        "relevant_chunk_ids": ["chunk_00005"]  # update with actual ID
    },
]

print(f"Eval set has {len(EVAL_SET)} labeled queries")

In [ ]:
# Compute hit-rate
def compute_hit_rate(eval_set, collection, k=3):
    """Compute hit-rate across eval set."""
    hits = 0
    total_relevant = 0
    results_log = []

    for item in eval_set:
        query = item["query"]
        relevant = set(item["relevant_chunk_ids"])

        query_results = collection.query(query_texts=[query], n_results=k)
        retrieved = set(query_results["ids"][0]) if query_results["ids"][0] else set()

        query_hits = len(retrieved & relevant)
        hits += query_hits
        total_relevant += len(relevant)

        results_log.append({
            "query": query[:50],
            "relevant": list(relevant),
            "retrieved": list(retrieved),
            "hit": query_hits > 0
        })

    hit_rate = hits / total_relevant if total_relevant > 0 else 0

    return {
        "hit_rate": round(hit_rate, 4),
        "hits": hits,
        "total": total_relevant,
        "per_query": results_log
    }


eval_result = compute_hit_rate(EVAL_SET, collection, k=3)

print(f"Hit-Rate: {eval_result['hit_rate']:.2%} ({eval_result['hits']}/{eval_result['total']})")
print(f"\nPer-query results:")
for r in eval_result["per_query"]:
    status = "HIT" if r["hit"] else "MISS"
    print(f"  [{status}] {r['query']}")

---

## Section 9: Chunk Size Experiment

### Lab Task: Compare Hit-Rate Across Chunk Sizes

Run the same evaluation queries with three different chunk sizes and record your results.

**Hypothesis:** Which size will perform best for policy FAQ content? Write your prediction before running.

In [ ]:
# YOUR HYPOTHESIS (fill in before running):
# I predict chunk size ___ will perform best because ___

experiment_configs = [
    {"chunk_size": 256, "overlap": 25},
    {"chunk_size": 512, "overlap": 50},
    {"chunk_size": 1024, "overlap": 100},
]

experiment_results = []

for config in experiment_configs:
    size = config["chunk_size"]
    overlap = config["overlap"]

    print(f"\n--- Experiment: chunk_size={size}, overlap={overlap} ---")

    # Chunk documents
    exp_splitter = RecursiveCharacterTextSplitter(
        chunk_size=size,
        chunk_overlap=overlap,
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    exp_chunks = exp_splitter.split_documents(documents)
    print(f"  Chunks: {len(exp_chunks)}")

    # Create new collection
    try:
        chroma_client.delete_collection(f"exp_{size}")
    except:
        pass
    exp_collection = chroma_client.create_collection(
        name=f"exp_{size}",
        metadata={"hnsw:space": "cosine"}
    )

    # Index
    exp_texts = [c.page_content for c in exp_chunks]
    exp_ids = [f"chunk_{i:05d}" for i in range(len(exp_chunks))]
    exp_metas = [c.metadata for c in exp_chunks]

    emb_response = client.embeddings.create(input=exp_texts, model="text-embedding-3-small")
    exp_embeddings = [item.embedding for item in emb_response.data]

    exp_collection.add(documents=exp_texts, embeddings=exp_embeddings, ids=exp_ids, metadatas=exp_metas)

    # For this experiment, we evaluate by semantic match rather than exact ID match
    # (since chunk IDs change per configuration)
    # Proxy evaluation: check if top result's content overlaps with expected section keywords
    section_keywords = {
        "How many annual leave days do full-time employees get?": ["annual leave", "20 days"],
        "Do I need a medical certificate for sick leave?": ["medical certificate", "sick leave"],
        "How long is maternity leave?": ["maternity", "26 weeks"],
        "What is the notice period for resignation?": ["notice", "resignation", "weeks"],
        "What are the steps in the disciplinary procedure?": ["disciplinary", "warning"],
        "Can I work from home and how many days per week?": ["remote work", "2 days"],
    }

    correct = 0
    for query, keywords in section_keywords.items():
        r = exp_collection.query(query_texts=[query], n_results=1)
        top_doc = r["documents"][0][0].lower() if r["documents"][0] else ""
        if any(kw.lower() in top_doc for kw in keywords):
            correct += 1

    hit_rate = correct / len(section_keywords)
    print(f"  Keyword hit-rate: {hit_rate:.2%} ({correct}/{len(section_keywords)})")

    experiment_results.append({
        "chunk_size": size,
        "overlap": overlap,
        "num_chunks": len(exp_chunks),
        "hit_rate": round(hit_rate, 4)
    })

print("\n" + "=" * 50)
print("EXPERIMENT RESULTS:")
print(f"{'Size':>6}  {'Overlap':>7}  {'Chunks':>6}  {'Hit-Rate':>9}")
for r in experiment_results:
    print(f"{r['chunk_size']:>6}  {r['overlap']:>7}  {r['num_chunks']:>6}  {r['hit_rate']:>9.2%}")

---

## Section 10: Edge Case Handling

### Concept: Robust Pipelines Handle the Unexpected

Production systems encounter:
- Empty or whitespace-only queries
- Very long queries
- Out-of-scope questions with no relevant documents
- API failures

Handle these explicitly rather than letting the pipeline silently fail.

In [ ]:
# Test edge cases
edge_cases = [
    "",                                               # Empty query
    "   ",                                            # Whitespace only
    "What is the salary and pay grade structure?",    # Out of scope
    "leave " * 100,                                   # Excessively long
]

print("Edge case handling results:")
print("-" * 60)

for query in edge_cases:
    display = repr(query[:40])
    result = rag_query(query, collection)
    print(f"Query: {display}")
    print(f"Answer: {result['answer'][:100]}")
    print()

---

## Section 11: Structured Logging

### Concept: Observability in AI Pipelines

For production pipelines, log everything you'd want for debugging:
- What was queried
- What was retrieved (IDs and scores)
- How long it took
- Whether the confidence threshold was met

In [ ]:
import json
from datetime import datetime

query_log = []

def logged_rag_query(question: str, collection) -> dict:
    """RAG query with structured logging."""
    start = time.time()
    result = rag_query(question, collection)
    elapsed = round((time.time() - start) * 1000, 2)

    log_entry = {
        "timestamp": datetime.now().isoformat(),
        "query": question,
        "answered": bool(result["sources"]),
        "num_sources": len(result["sources"]),
        "top_score": result["sources"][0]["score"] if result["sources"] else None,
        "latency_ms": elapsed
    }
    query_log.append(log_entry)
    return result


# Run a few queries with logging
test_queries = [
    "How many annual leave days do I get?",
    "What are the core hours for remote work?",
    "What is the pension contribution rate?",  # out of scope
]

for q in test_queries:
    logged_rag_query(q, collection)

print("Query log:")
print(json.dumps(query_log, indent=2))

---

## Key Takeaways

1. **RAG = Index + Retrieve + Generate** — each phase has distinct design choices
2. **Embeddings capture meaning as numbers** — similar text → nearby vectors → semantic search works
3. **Chunk size is your highest-impact decision** — measure hit-rate to find the sweet spot
4. **Always measure before optimizing** — hit-rate gives you an objective baseline
5. **Production pipelines need guards** — empty queries, low scores, API failures happen in the real world
6. **Log everything from day one** — you'll thank yourself when debugging at 2am

---

## Next Steps

**Module 7: Building RAG — Day 2**
- Query expansion: HyDE and multi-query retrieval
- Re-ranking with cross-encoders
- Evaluation frameworks: RAGAS and LLM-as-judge
- Moving to Pinecone for production scale

**Take-Home Extension:**
- Implement paragraph-based chunking (split on `\n\n`)
- Re-index the policy document and compare hit-rate
- Try MMR retrieval: `search_type='mmr'` in LangChain's ChromaDB wrapper

In [ ]:
# Cleanup
policy_path.unlink(missing_ok=True)
print("✓ Cleanup complete. Well done — you've built your first RAG pipeline!")